# Homework 10

Ryan Mersereau

## Part 1 - Creating Streaming Data Using `rate`

Import necessary libraries and create the Spark session.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sqrt, pow

spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 16:49:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/21 16:49:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/21 16:49:26 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/21 16:49:26 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/21 16:49:26 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


## Step 1: Read in the stream using the `rate` format

We'll set up a data stream using the `rate` format, which automatically generates rows with a `timestamp` and an incrementing integer `value` column.

In [2]:
rateDF = spark.readStream.format("rate").load()
rateDF

DataFrame[timestamp: timestamp, value: bigint]

## Step 2: Set up transformations

Prior to starting the stream, we'll set up the sequence of transformations on the rate `value` column:
- **Square root** of the rate `value` using `sqrt()`
- **Mod 4** of the rate `value` using the modulo operator (`%`)

In [3]:
manipDF = rateDF \
    .withColumn("sqrt_value", sqrt(col("value"))) \
    .withColumn("mod4_value", col("value") % 4)

## Step 3 & 4: Write stream to `memory` and `.start()` the query

We'll write the output to memory format so we can query it with SQL. We give the table a name with `queryName()` and start the stream with `.start()`.

We'll let it run for ~30 seconds, then stop and query the table.

In [4]:
writeTable = manipDF \
    .writeStream \
    .format("memory") \
    .queryName("rate_table") \
    .start()

26/04/21 16:49:39 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-8f36e1e5-6844-42ec-ad17-5c9eb1c34a30. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/21 16:49:39 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Let the stream run for about 30 seconds before stopping...

In [5]:
import time
time.sleep(30)

In [6]:
writeTable.stop()

26/04/21 16:50:20 WARN DAGScheduler: Failed to cancel job group 79581898-bcb2-41d2-ab7b-9117036fe715. Cannot find active jobs for it.
26/04/21 16:50:20 WARN DAGScheduler: Failed to cancel job group 79581898-bcb2-41d2-ab7b-9117036fe715. Cannot find active jobs for it.


## Step 5: Query the in-memory table

Now that the query has been stopped, we can output the entire table using `spark.sql()`.

In [7]:
spark.sql("select * from rate_table").show()

+--------------------+-----+------------------+----------+
|           timestamp|value|        sqrt_value|mod4_value|
+--------------------+-----+------------------+----------+
|2026-04-21 16:49:...|    0|               0.0|         0|
|2026-04-21 16:49:...|    1|               1.0|         1|
|2026-04-21 16:49:...|    2|1.4142135623730951|         2|
|2026-04-21 16:49:...|    3|1.7320508075688772|         3|
|2026-04-21 16:49:...|    4|               2.0|         0|
|2026-04-21 16:49:...|    5|  2.23606797749979|         1|
|2026-04-21 16:49:...|    6| 2.449489742783178|         2|
|2026-04-21 16:49:...|    7|2.6457513110645907|         3|
|2026-04-21 16:49:...|    8|2.8284271247461903|         0|
|2026-04-21 16:49:...|    9|               3.0|         1|
|2026-04-21 16:49:...|   10|3.1622776601683795|         2|
|2026-04-21 16:49:...|   11|   3.3166247903554|         3|
|2026-04-21 16:49:...|   12|3.4641016151377544|         0|
|2026-04-21 16:49:...|   13| 3.605551275463989|         

## Part 2: Using data from csv with a pipeline

### Step 1: Imports, Read fit csv as a Spark SQL Dataframe

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, VectorAssembler
from pyspark.ml import Pipeline

spark = SparkSession.builder.getOrCreate()

bikeDF = spark.read.csv("bikeDetails_for_fit.csv", header=True, inferSchema=True)
bikeDF.show(5)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 17:12:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+--------------------+-------------+----+-----------+---------+---------+-----------------+
|                name|selling_price|year|seller_type|    owner|km_driven|ex_showroom_price|
+--------------------+-------------+----+-----------+---------+---------+-----------------+
|Royal Enfield Cla...|       175000|2019| Individual|1st owner|      350|             NULL|
|           Honda Dio|        45000|2017| Individual|1st owner|     5650|             NULL|
|Royal Enfield Cla...|       150000|2018| Individual|1st owner|    12000|           148114|
|Yamaha Fazer FI V...|        65000|2015| Individual|1st owner|    23000|            89643|
|Yamaha SZ [2013-2...|        20000|2011| Individual|2nd owner|    21000|             NULL|
+--------------------+-------------+----+-----------+---------+---------+-----------------+
only showing top 5 rows


### Step 2: Set up `SQLTransformer` 

This will perform our required log transformations, variable renaming, and creation of dummy variable.

In [2]:
sqlTrans = SQLTransformer(statement="""
    SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven,
    CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner
    FROM __THIS__
""")

### Step 3: Set up `VectorTransformer` 

This is done to created a `features` column with `year`, `log_km_driven`, and `one_owner` variables.

In [3]:
assembler = VectorAssembler(
    inputCols=["year", "log_km_driven", "one_owner"],
    outputCol="features"
)

### Step 4: Fitting the `Pipeline`

The pipeline includes the two steps above, and then can by fit to the SQL data frame and saved as an object.

In [4]:
pipeline = Pipeline(stages=[sqlTrans, assembler])
fittedPipeline = pipeline.fit(bikeDF)

### Step 5: Set up read stream to watch for csv files

We created an **empty** folder in our jupyter environment called **bike_stream_folder**. This cell sets up the stream and tells spark to anticipate files within this folder. **Note: We don't want to put the .csv files
within this folder prior to starting the stream (next step)**. The point of this is to replicate real streaming data by adding the files to this targeted folder while the stream is running.

In [5]:
bikeStream = spark.readStream \
    .schema(bikeDF.schema) \
    .option("header", True) \
    .csv("bike_stream_folder")

### Step 6: Apply fitted pipeline and start the stream

In [6]:
transformedStream = fittedPipeline.transform(bikeStream)

streamQuery = transformedStream \
    .writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

26/04/21 17:27:23 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-85dcb629-585b-46ad-bb1e-62ddbf4946df. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/21 17:27:23 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
| 8.987196820661973|2003|10.887436932884098|        1|[2003.0,10.887436...|
|11.156250521031495|2018| 9.615805480084347|        1|[2018.0,9.6158054...|
|10.819778284410283|2016| 8.987196820661973|        1|[2016.0,8.9871968...|
| 10.46310334047155|2015|10.582738627903963|        1|[2015.0,10.582738...|
| 9.903487552536127|2006|11.225243392518447|        1|[2006.0,11.225243...|
|10.819778284410283|2012|10.239959789157341|        1|[2012.0,10.239959...|
| 10.51867319162636|2008| 11.03488966402723|        1|[2008.0,11.034889...|
|11.141861783579396|2018| 9.392661928770137|        1|[2018.0,9.3926619...|
|10.239959789157341|2012| 10.81975828421028|        1|[2012.0,10.81

Now, we can start placing the data files into the bike_stream_folder one by one. These should be printed to the console each time a new file lands.

### Step 7: End the stream once all 5 files have been added

In [7]:
streamQuery.stop()

26/04/21 17:27:57 WARN DAGScheduler: Failed to cancel job group d692c448-ad7d-45aa-963d-332566a91b8e. Cannot find active jobs for it.
26/04/21 17:27:57 WARN DAGScheduler: Failed to cancel job group d692c448-ad7d-45aa-963d-332566a91b8e. Cannot find active jobs for it.
